In [1]:
import torch as t
from torch import Tensor
from jaxtyping import Float
from tqdm import tqdm
import numpy as np

from nnsight.models.UnifiedTransformer import UnifiedTransformer

from transformer_lens import HookedTransformer

device = "cuda"

In [2]:
model = UnifiedTransformer(
    'gpt2-small',
    processing=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    device=device,
)
tokenizer = model.tokenizer

model.set_use_hook_mlp_in(True)
model.set_use_split_qkv_input(True)
model.set_use_attn_result(True)

Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
from ioi_dataset import IOIDataset

N = 25
clean_dataset = IOIDataset(
    prompt_type='mixed',
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
    seed=1,
    device=device
)
corr_dataset = clean_dataset.gen_flipped_prompts('ABC->XYZ, BAB->XYZ')

In [4]:
def ave_logit_diff(
    logits: Float[Tensor, 'batch seq d_vocab'],
    ioi_dataset: IOIDataset,
    per_prompt: bool = False
):
    '''
        Return average logit difference between correct and incorrect answers
    '''
    # Get logits for indirect objects
    batch_size = logits.size(0)
    io_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.io_tokenIDs[:batch_size]]
    s_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.s_tokenIDs[:batch_size]]
    # Get logits for subject
    logit_diff = io_logits - s_logits
    return logit_diff if per_prompt else logit_diff.mean()


with t.no_grad():
    with model.trace(clean_dataset.toks):
        clean_logits = model.output.save()

    with model.trace(corr_dataset.toks):
        corrupt_logits = model.output.save()

clean_logits = clean_logits.value
corrupt_logits = corrupt_logits.value

clean_logit_diff = ave_logit_diff(clean_logits, clean_dataset).item()
corrupt_logit_diff = ave_logit_diff(corrupt_logits, corr_dataset).item()

def ioi_metric(
    logits: Float[Tensor, "batch seq_len d_vocab"],
    corrupted_logit_diff: float = corrupt_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
    ioi_dataset: IOIDataset = clean_dataset
 ):
    patched_logit_diff = ave_logit_diff(logits, ioi_dataset)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

def negative_ioi_metric(logits: Float[Tensor, "batch seq_len d_vocab"]):
    return -ioi_metric(logits)
    
# Get clean and corrupt logit differences
with t.no_grad():
    clean_metric = ioi_metric(clean_logits, corrupt_logit_diff, clean_logit_diff, clean_dataset)
    corrupt_metric = ioi_metric(corrupt_logits, corrupt_logit_diff, clean_logit_diff, corr_dataset)

print(f'Clean direction: {clean_logit_diff}, Corrupt direction: {corrupt_logit_diff}')
print(f'Clean metric: {clean_metric}, Corrupt metric: {corrupt_metric}')

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Clean direction: 2.805180311203003, Corrupt direction: 1.7103774547576904
Clean metric: 1.0, Corrupt metric: 0.0


In [5]:
import eap

import importlib

importlib.reload(eap)

graph = eap.EAP(model.cfg, components=["head", "mlp"])

In [6]:
graph.run(
    model,
    clean_dataset.toks,
    corr_dataset.toks,
    batch_size=25,
    metric=ioi_metric,
)
# edges = graph.top_edges(n=20, format=True)

tensor([[[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          ...,
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00]],

         [[ 1.7449e-05, -2.4298e-05, -1.7559e-05,  ...,  3.7654e-05,
            2.7946e-05, -1.0273e-04],
          [ 1.0924e-06, -1.0150e-06, -5.5358e-07,  ..., -9.6681e-07,
           -4.1969e-06, -3.3409e-08],
          [ 6.9858e-04,  2.9016e-04, -1.0023e-04,  ..., -1.1672e-03,
            5.5731e-04, -7.9734e-04],
          ...,
     

In [7]:
edges = graph.top_edges(n=20, format=True)

head.1.4 -> [0.024] -> head.6.5.q
head.1.4 -> [-0.024] -> head.8.10.q
head.1.4 -> [0.023] -> head.8.10.k
head.5.9 -> [0.017] -> head.6.6.v
head.1.3 -> [-0.017] -> head.6.5.q
head.1.4 -> [-0.017] -> head.7.9.q
head.1.4 -> [-0.017] -> head.6.5.k
head.1.3 -> [0.015] -> head.8.10.q
head.1.8 -> [-0.015] -> head.11.10.k
head.1.4 -> [0.014] -> head.7.9.k
head.1.4 -> [0.014] -> head.9.7.k
head.2.10 -> [-0.013] -> head.8.10.q
head.1.3 -> [0.013] -> head.6.5.k
head.1.4 -> [-0.013] -> head.9.7.q
head.5.5 -> [-0.013] -> head.6.6.v
head.0.3 -> [0.013] -> head.6.5.q
head.1.3 -> [0.012] -> head.7.9.q
head.1.9 -> [0.012] -> head.6.5.k
head.1.8 -> [-0.012] -> head.8.10.k
head.1.9 -> [-0.012] -> head.6.5.q
